# Exploratory Data Analysis


In [1]:
import sys
from pathlib import Path


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    """Walk upward from the current working dir to locate the repo root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} above {here}")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # works even without `pip install -e .`

DATA_PATH = REPO_ROOT / "data" / "raw" / "flights.csv"
FIG_DIR = REPO_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root :", REPO_ROOT)
print("Data path :", DATA_PATH, "| exists:", DATA_PATH.exists())
print("Figures   :", FIG_DIR)

Repo root : /path/to/flight-delay-prediction
Data path : /path/to/flight-delay-prediction/data/raw/flights.csv | exists: True
Figures   : /path/to/flight-delay-prediction/reports/figures


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from flight_delay.data.load import load_flights
from flight_delay.data.pipeline import build_data_pipeline
from flight_delay.features.selection import (
    EXCLUDED_COLUMNS,
    TARGET_COLUMN,
    split_features_and_target,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_SEED = 42


def save_fig(name: str) -> None:
    """Save the current figure to reports/figures/ as a labeled PNG."""
    plt.tight_layout()
    out = FIG_DIR / f"{name}.png"
    plt.savefig(out, bbox_inches="tight")
    print("saved:", out.relative_to(REPO_ROOT))

##  Here, we loaded the data

`raw` is the full CSV (for data-quality checks). The pipeline computes timetable congestion
counts before dropping rows with undefined targets, then builds target, temporal, route and
prior-date historical features before the chronological split. Historical outcome availability
at the prediction instant is not verified.

In [3]:
raw = load_flights(DATA_PATH)
print("raw shape:", raw.shape)
raw.head()

raw shape: (336776, 21)


,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,...,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.


In [4]:
train, validation, test = build_data_pipeline(DATA_PATH)
print("train:", train.shape, "| validation:", validation.shape, "| test:", test.shape)
print("\nColumns after pipeline:\n", list(train.columns))

train: (228299, 45) | validation: (50466, 45) | test: (48581, 45)

Columns after pipeline:
 ['id', 'year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour', 'name', 'origin_hourly_departures', 'dest_hourly_arrivals', 'is_delayed', 'day_of_week', 'is_weekend', 'season', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'route', 'distance_band', 'is_long_haul', 'distance_log', 'carrier_historical_delay_rate', 'carrier_historical_flight_count', 'origin_historical_delay_rate', 'origin_historical_flight_count', 'destination_historical_delay_rate', 'destination_historical_flight_count', 'route_historical_delay_rate', 'route_historical_flight_count']


## Data quality, dtypes & missing values

In [5]:
missing = raw.isna().sum()
missing_pct = (missing / len(raw) * 100).round(2)
quality = pd.DataFrame({"missing": missing, "missing_%": missing_pct})
quality = quality[quality["missing"] > 0].sort_values("missing", ascending=False)
print("Columns with missing values:")
quality

Columns with missing values:


,missing,missing_%
arr_delay,9430,2.80
air_time,9430,2.80
arr_time,8713,2.59
dep_time,8255,2.45
dep_delay,8255,2.45
tailnum,2512,0.75


In [6]:
# arr_delay drives the target; rows with missing arr_delay are dropped by the pipeline.
n_missing_arr = int(raw["arr_delay"].isna().sum())
print(f"Rows with missing arr_delay (cancelled/diverted-like): {n_missing_arr:,} "
      f"({n_missing_arr / len(raw) * 100:.2f}%)")
print(f"Rows kept for modelling: {len(train) + len(validation) + len(test):,}")

Rows with missing arr_delay (cancelled/diverted-like): 9,430 (2.80%)
Rows kept for modelling: 327,346


## Target base rate

`is_delayed = 1` when `arr_delay >= 15 min`. The base rate sets the baseline a
model must beat and tells us how imbalanced the problem is.

In [7]:
base_rate = train[TARGET_COLUMN].mean()
print(f"Train base rate (P[delayed]): {base_rate:.4f}  ({base_rate*100:.2f}%)")
print(f"Class balance: {train[TARGET_COLUMN].value_counts().to_dict()}")

ax = train[TARGET_COLUMN].value_counts().sort_index().plot(
    kind="bar", color=["#4C72B0", "#DD8452"]
)
ax.set_xticklabels(["on-time (0)", "delayed (1)"], rotation=0)
ax.set_ylabel("flights")
ax.set_title("Target distribution (train)")
save_fig("target_distribution")
plt.show()

Train base rate (P[delayed]): 0.2591  (25.91%)
Class balance: {0: 169139, 1: 59160}
saved: reports/figures/target_distribution.png


## Delay rate by carrier

In [8]:
by_carrier = (
    train.groupby("carrier")[TARGET_COLUMN]
    .agg(delay_rate="mean", flights="count")
    .sort_values("delay_rate", ascending=False)
)
display(by_carrier)

ax = by_carrier["delay_rate"].plot(kind="bar", color="#4C72B0")
ax.axhline(base_rate, color="red", ls="--", label=f"base rate ({base_rate:.2f})")
ax.set_ylabel("delay rate")
ax.set_title("Delay rate by carrier (train)")
ax.legend()
save_fig("delay_by_carrier")
plt.show()

,delay_rate,flights
carrier,,
F9,0.405983,468
OO,0.357143,14
YV,0.356164,365
EV,0.344887,35197
FL,0.338798,2379
B6,0.291105,38213
MQ,0.290743,17586
9E,0.276645,11625
WN,0.245763,8260


saved: reports/figures/delay_by_carrier.png


## Delay rate by scheduled hour and by month

Temporal structure motivates the cyclic encoding of `hour` and `month`.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

by_hour = train.groupby("hour")[TARGET_COLUMN].mean()
by_hour.plot(ax=axes[0], marker="o", color="#4C72B0")
axes[0].axhline(base_rate, color="red", ls="--")
axes[0].set(title="Delay rate by scheduled hour", xlabel="hour", ylabel="delay rate")

by_month = train.groupby("month")[TARGET_COLUMN].mean()
by_month.plot(ax=axes[1], marker="o", color="#DD8452")
axes[1].axhline(base_rate, color="red", ls="--")
axes[1].set(title="Delay rate by month", xlabel="month", ylabel="delay rate")

save_fig("delay_by_hour_and_month")
plt.show()

saved: reports/figures/delay_by_hour_and_month.png


## Delay rate by origin and by top routes

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

by_origin = train.groupby("origin")[TARGET_COLUMN].mean().sort_values(ascending=False)
by_origin.plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].axhline(base_rate, color="red", ls="--")
axes[0].set(title="Delay rate by origin", ylabel="delay rate")
axes[0].tick_params(axis="x", rotation=0)

route = train["origin"].astype(str) + ">" + train["dest"].astype(str)
top_routes = route.value_counts().head(15).index
route_rate = (
    train.assign(route=route)[lambda d: d["route"].isin(top_routes)]
    .groupby("route")[TARGET_COLUMN].mean().sort_values(ascending=False)
)
route_rate.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].axvline(base_rate, color="red", ls="--")
axes[1].set(title="Delay rate — 15 busiest routes", xlabel="delay rate")

save_fig("delay_by_origin_and_route")
plt.show()

saved: reports/figures/delay_by_origin_and_route.png


## Distance vs. delay

In [11]:
dist_bins = pd.cut(train["distance"], bins=[0, 250, 500, 750, 1000, 1500, 2000, 6000])
by_dist = train.groupby(dist_bins, observed=True)[TARGET_COLUMN].mean()
ax = by_dist.plot(kind="bar", color="#4C72B0")
ax.axhline(base_rate, color="red", ls="--", label=f"base rate ({base_rate:.2f})")
ax.set(title="Delay rate by distance band", xlabel="distance (miles)", ylabel="delay rate")
ax.tick_params(axis="x", rotation=30)
ax.legend()
save_fig("delay_by_distance")
plt.show()

saved: reports/figures/delay_by_distance.png


## Cyclic time-encoding sanity check

Hour and month are cyclic (hour 23 is close to hour 0). A sin/cos encoding keeps
that adjacency. This cell demonstrates the transform the modelers should apply
(documented in `docs/feature_policy.md`).

In [12]:
def add_cyclic(df: pd.DataFrame, col: str, period: int) -> pd.DataFrame:
    out = df.copy()
    out[f"{col}_sin"] = np.sin(2 * np.pi * out[col] / period)
    out[f"{col}_cos"] = np.cos(2 * np.pi * out[col] / period)
    return out


demo = add_cyclic(train[["hour"]].drop_duplicates().sort_values("hour"), "hour", 24)
ax = demo.plot.scatter(x="hour_sin", y="hour_cos")
for _, r in demo.iterrows():
    ax.annotate(int(r["hour"]), (r["hour_sin"], r["hour_cos"]), fontsize=7)
ax.set_title("Cyclic encoding of hour (sin vs cos)")
ax.set_aspect("equal")
save_fig("cyclic_hour_encoding")
plt.show()

saved: reports/figures/cyclic_hour_encoding.png


## Historical features & correlation with target

The pipeline adds prior-calendar-date historical delay rates. Their ordering excludes today's
and future dates, but exact availability at a prediction instant is not verified. Check
whether they carry signal.

In [13]:
hist_cols = [c for c in train.columns if "historical_delay_rate" in c]
corr = train[hist_cols + [TARGET_COLUMN]].corr()[TARGET_COLUMN].drop(TARGET_COLUMN)
print("Correlation of historical delay-rate features with target:")
print(corr.sort_values(ascending=False))

ax = corr.sort_values().plot(kind="barh", color="#4C72B0")
ax.set_title("Historical feature correlation with is_delayed (train)")
save_fig("historical_feature_correlation")
plt.show()

Correlation of historical delay-rate features with target:
carrier_historical_delay_rate        0.119100
route_historical_delay_rate          0.115419
destination_historical_delay_rate    0.090493
origin_historical_delay_rate         0.042705
Name: is_delayed, dtype: float64


saved: reports/figures/historical_feature_correlation.png


## Feature selection is **two stages** (important)

`split_features_and_target` is only the **first** gate. It removes the target,
post-departure columns and obvious identifiers, but its output still contains
non-predictor columns such as `flight`, `tailnum` and `time_hour`. With the two
schedule-based congestion columns, this candidate frame contains 36 columns.
The models do **not** train on this frame directly.

Each model then applies a **second, stricter gate** — the shared
`select_model_features` allowlist in `features/model_features.py`. The canonical
frozen evaluation uses the 31-feature `full` set, which excludes congestion.
The `full_congestion` set adds two schedule-derived proxies (33 features) and is
used only in the validation ablation, not the held-out test. Both sets exclude
`flight`, `tailnum` and `time_hour` from model inputs.

In [14]:
X_train, y_train = split_features_and_target(train)
print("Excluded (leakage / id / redundant):", sorted(EXCLUDED_COLUMNS))
print("\nFeature columns (", X_train.shape[1], "):")
for c in X_train.columns:
    print("  -", c, "|", X_train[c].dtype)
print("\nNote: `flight`, `tailnum`, `dest`, `carrier`, `origin`, `time_hour` are still")
print("non-numeric/high-cardinality — modelers must encode or drop them in their pipelines.")

Excluded (leakage / id / redundant): ['air_time', 'arr_delay', 'arr_time', 'dep_delay', 'dep_time', 'id', 'is_delayed', 'name', 'year']

Feature columns ( 36 ):
  - month | int64
  - day | int64
  - sched_dep_time | int64
  - sched_arr_time | int64
  - carrier | str
  - flight | int64
  - tailnum | str
  - origin | str
  - dest | str
  - distance | int64
  - hour | int64
  - minute | int64
  - time_hour | str
  - origin_hourly_departures | int32
  - dest_hourly_arrivals | int32
  - day_of_week | int8
  - is_weekend | int8
  - season | int8
  - hour_sin | float64
  - hour_cos | float64
  - month_sin | float64
  - month_cos | float64
  - day_of_week_sin | float64
  - day_of_week_cos | float64
  - route | str
  - distance_band | category
  - is_long_haul | int8
  - distance_log | float64
  - carrier_historical_delay_rate | float64
  - carrier_historical_flight_count | int64
  - origin_historical_delay_rate | float64
  - origin_historical_flight_count | int64
  - destination_historical_del

## Takeaways for the team

- Target base rate and class imbalance are quantified above — use `scale_pos_weight`
  (XGBoost) / `class_weight` (LR) and report **Average Precision (AP)**, not just accuracy.
- Clear signal in **carrier, hour, month, origin/route** → justifies the engineered
  features and the cyclic time encoding.
- Historical delay-rate features correlate with the target → keep them.
- Dataset has only **3 origins (NYC)** — fine for Phase 1; flag for the DL/embedding
  phase and the viva.
- Figures saved in `reports/figures/` for the report and slides.